# RO change over time

## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## specify args

In [ ]:
## should we remove median?
REMOVE_MEDIAN = True

## Functions

In [ ]:
def get_rolling_std(data, n=20):
    """
    Get standard deviation, computing over time and ensemble member. To increase
    sample size for variance estimate, compute over time window of 2n+1
    years, centered at given year.
    """

    ## do the computation
    kwargs = dict(fn=np.std, n=n, reduce_ensemble_dim=False)
    data_std = src.utils.get_rolling_fn_bymonth(data, **kwargs)

    ## unstack year and month
    data_std = src.utils.unstack_month_and_year(data_std)

    return data_std


def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits


def get_fits_over_time_wrapper(
    data_rolling, model, by_member=False, fname=None, **fit_kwargs
):
    """wrapper function to handle saving/loading"""

    ## function to compute fits
    get_fits = lambda: get_fits_over_time(
        data_rolling, model=model, by_member=by_member, **fit_kwargs
    )

    ## if fname not specified, compute without loading/saving
    if fname is None:
        fits = get_fits()

    else:

        ## full path to file
        fp = pathlib.Path(os.environ["SAVE_FP"], "fits_cesm", fname)

        ## load if it already exists
        if fp.is_file():
            fits = xr.open_dataset(fp)

        ## otherwise, compute and save
        else:
            fits = get_fits()
            fits.to_netcdf(fp)

    return fits


def get_sims_over_time(model, params, **simulation_kwargs):
    """Compute stats over time"""

    ## take ensemble mean if necessary
    if "member" in params.dims:
        params = params.mean("member")

    ## empty list to hold result
    sims = []

    ## loop through years
    for y in tqdm.tqdm(params.year):

        ## do simulation
        sim_y = model.simulate(fit_ds=params.sel(year=y), **simulation_kwargs)

        ## append
        sims.append(sim_y)

    ## put back in xarray
    sims = xr.concat(sims, dim=params.year)

    return sims


def save(fig, fname, dpi=800, do_save=False):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "pre-egu-updates")

    ## get fname
    fname = f"{fname}-{varnames[0]}-subtract_median_{REMOVE_MEDIAN}.pdf"

    if do_save:
        fig.savefig(save_dir / fname, dpi=dpi, format="pdf")

    return

## Load data

### T, h

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True, load_h_cust=True, max_grad=True)

#### Load ELI

In [ ]:
## Load ELI
eli_all = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/eli.nc"))
eli_forced, eli = src.utils.separate_forced(eli_all)

## add to data
Th = xr.merge([Th, eli])

Scale it

#### Merged Niño index

In [ ]:
def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])

def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])

def get_ddx(x):
    """compute horizontal gradient"""

    ## ranges
    lat_range = dict(latitude=slice(-5, 5))
    lon_range_w = dict(longitude=slice(120, 160))
    lon_range_e = dict(longitude=slice(210, 270))

    ## spatial avg
    avg = lambda x: x.mean(["latitude", "longitude"])

    return avg(x.sel(**lon_range_w, **lat_range)) - avg(
        x.sel(**lon_range_e, **lat_range)
    )


## spatial data
forced, anom = src.utils.load_consolidated()
sst_total = xr.merge([forced["sst"] + anom["sst"], forced["sst_comp"]])
sst_anom = anom[["sst","sst_comp"]]

## load tropical SST avg
Ttrop = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/trop_sst.nc")).mean("member")

## compute anoms
Th["T_e"] = src.utils.reconstruct_wrapper(sst_anom, get_Te)["sst"]
Th["T_w"] = src.utils.reconstruct_wrapper(sst_anom, get_Tw)["sst"]

## compute totals
T_e_total = src.utils.reconstruct_wrapper(sst_total, get_Te)["sst"]
T_w_total = src.utils.reconstruct_wrapper(sst_total, get_Tw)["sst"]
T_3_total = src.utils.reconstruct_wrapper(sst_total, src.utils.get_nino3)["sst"]
T_34_total = src.utils.reconstruct_wrapper(sst_total, src.utils.get_nino34)["sst"]

## relative SST
Th["T_e_rel"] = T_e_total - Ttrop["trop_sst_15"]
Th["T_w_rel"] = T_w_total - Ttrop["trop_sst_15"]
Th["T_3_rel"] = T_3_total - Ttrop["trop_sst_15"]
Th["T_34_rel"] = T_34_total - Ttrop["trop_sst_15"]
Th["T_3_tot"] = T_3_total
Th["T_34_tot"] = T_34_total

## compute dTdx
Th["dTdx_total"] = src.utils.reconstruct_wrapper(
    forced[["sst", "sst_comp"]],
    get_ddx,
)["sst"]

### NHF

In [ ]:
## get NHF in Niño regions
nhf_e = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=get_Te
)["nhf"]
nhf_w = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=get_Tw
)["nhf"]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q_e = nhf_e * sec_per_yr / (rho * Cp * H)
Q_w = nhf_w * sec_per_yr / (rho * Cp * H)

## merge with Th data
Th["Q_e"] = Q_e.sel(time=Th.time)
Th["Q_w"] = Q_w.sel(time=Th.time)

#### zonal gradient

In [ ]:
def get_ddx(x):
    """compute horizontal gradient"""

    ## ranges
    lat_range = dict(latitude=slice(-5, 5))
    lon_range_w = dict(longitude=slice(120, 160))
    lon_range_e = dict(longitude=slice(210, 270))

    ## spatial avg
    avg = lambda x: x.mean(["latitude", "longitude"])

    return avg(x.sel(**lon_range_w, **lat_range)) - avg(
        x.sel(**lon_range_e, **lat_range)
    )

In [ ]:
## compute dTdx
Th["dTdx_total"] = src.utils.reconstruct_wrapper(
    forced[["sst", "sst_comp"]],
    get_ddx,
)["sst"]

### preprocess

In [ ]:
## standardize (for convenience)
Th /= Th.std()

## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

## drop last year because of NaNs
Th_rolling = Th_rolling.sel(year=slice(None, 2080))

## compute grad
Th_rolling["dTdx"] = Th_rolling["T_3"] - Th_rolling["T_4"]

#### eli

In [ ]:
# compute climatological gradient
dTdx_clim = Th_rolling["dTdx_total"].groupby("time.month").mean(["time"])

## get scaling factor
dTdx_scale = dTdx_clim / dTdx_clim.max()

## apply scaling
for v in list(Th_rolling):
    if "eli" in v:
        Th_rolling[f"{v}_scaled"] = Th_rolling[v].groupby("time.month") * dTdx_scale

        ## subtrack median
        grouped = Th_rolling[f"{v}_scaled"].groupby("time.month")
        Th_rolling[f"{v}_scaled_median"] = grouped - grouped.median(["member", "time"])

#### Scatter plot

spatial structure of $T$

In [ ]:
Th_ = Th_rolling.resample({"time": "QS-JAN"}).mean().isel(time=slice(4, -4, 4))

fig, axs = plt.subplots(1, 2, figsize=(6,3), layout="constrained")
for ax, yr in zip(axs, [2010, 2080]):
    axs[0].scatter(
        Th_["T_e"].sel(year=yr),
        Th_["T_w"].sel(year=yr),
        s=3,
    )
    axs[1].scatter(
        Th_["T_e"].sel(year=yr),
        Th_["T_e"].sel(year=yr) - Th_["T_w"].sel(year=yr),
        s=3,
    )

src.utils.set_lims(axs)
axs[1].set_yticks([])
plt.show()

In [ ]:
Th_ = Th_rolling.resample({"time": "QS-JAN"}).mean().isel(time=slice(3, -4, 4))
# Th_ = Th_rolling.isel(time=slice(11, None, 12))

fig, axs = plt.subplots(2, 2, figsize=(6,6), layout="constrained")
for j, yr in enumerate([2010, 2080]):
    axs[j,0].scatter(
        # Th_["T_w_rel"].sel(year=yr),
        Th_["eli_05_scaled"].sel(year=yr),
        Th_["Q_w"].sel(year=yr),
        s=3,
    )
    axs[j,1].scatter(
        # Th_["T_e_rel"].sel(year=yr),
        Th_["eli_05_scaled"].sel(year=yr),
        Th_["Q_e"].sel(year=yr),
        s=3,
    )

## formatting
src.utils.set_lims(axs)
for ax in axs.flatten():
    ax_kwargs = dict(ls="--", c="k", lw=.8)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)
for ax in axs[:,1]:
    ax.set_yticks([])
for ax, t in zip(axs[0,:], [r"180-130$^{\circ}$W", r"130-80$^{\circ}$W"]):
    ax.set_xticks([])
    ax.set_title(t)
for ax in axs[:,0]:
    ax.set_ylabel(r"$Q$ (K yr$^{-1}$)")

for ax in axs[-1,:]:
    ax.set_xlabel("rel. SST (K)")
plt.show()

In [ ]:
# Th_ = Th_rolling.resample({"time": "QS-NOV"}).mean().isel(time=slice(4, -4, 4))
Th_ = Th_rolling.isel(time=slice(11, None, 12))

fig, axs = plt.subplots(1,2,figsize=(6,2), layout="constrained")

##
for ax, loc in zip(axs, ["w","e"]):

    ## edges for pdf
    edges = np.linspace(-4,2.5,15)

    ## loop thru years
    for y in [2010, 2080]:

        ## get data
        data_ = Th_[f"T_{loc}_rel"].sel(year=y).values.flatten()

        ## compute PDF
        pdf, _ = src.utils.get_empirical_pdf(data_, edges=edges)

        ## plot
        ax.stairs(pdf, edges, fill=None, lw=2, label=y)

src.utils.set_lims(axs)
axs[1].set_yticks([])
axs[0].set_title(r"180-130$^{\circ}$W")
axs[1].set_title(r"130-80$^{\circ}$W")
src.utils.add_vticks(axs, xticks=[-3,0,2],xlines=[0])
axs[0].legend()
plt.show()

In [ ]:
## orignal 
Th_res = xr.merge([Th, Ttrop]).resample({"time":"QS-DEC"}).mean().isel(time=slice(4,-4,4))
Th_q = Th_res.quantile(q=[.05,.1,.5,.9,.95],dim="member")
Th_q = Th_q.rolling({"time":31}, center=True, min_periods=1).mean()

is max temperature in springtime?

In [ ]:
v = "T_3_rel"

fig,axs = plt.subplots(1,2, figsize=(7,3))
# ax.plot(Th_q["T_3"].sel(quantile=.5))
for ax, y in zip(axs,[Th_q[v], np.abs((Th_q-Th_q.sel(quantile=.5))[v])]):
# for ax, y in zip(axs,[Th_q[v], np.abs((Th_q-Th_q.sel(quantile=.5))[v])]):
    
    ax.plot(y.time.dt.year, y.sel(quantile=.5), c="k")
    ax.plot(y.time.dt.year, y.sel(quantile=.1), label="La Niña")
    ax.plot(y.time.dt.year, y.sel(quantile=.9), label="El Niño")
    ax.axvline(2010, ls="--", c="k")
ax.legend()
plt.show()

#### scatter plot

In [ ]:
v0 = "trop_sst_15"
v1 = "T_3_rel"
delta = lambda x : x-x.isel(time=0)
sel = lambda x : delta(x)/delta(x).isel(time=-1)
fig,ax = plt.subplots(figsize=(4,4))
ax.set_aspect("equal")

ax.scatter(
    sel(Th_q[v0]), sel(Th_q[v1].sel(quantile=.5)), s=10, c="k",
)

ax.scatter(
    sel(Th_q[v0]), sel(Th_q[v1].sel(quantile=.05)), s=10,
)

ax.scatter(
    sel(Th_q[v0]), sel(Th_q[v1].sel(quantile=.95)), s=10,
)

for q in [.05,.95]:
    ax.scatter(
        sel(Th_q[v0]).isel(time=160), sel(Th_q[v1].sel(quantile=q)).isel(time=160), c="magenta",
    )

zz = np.linspace(0,1)
ax.plot(zz,zz,ls="--", c="gray", lw=1)
plt.show()

#### Try to reproduce PJ's plot

In [ ]:
avg_dims = ["time"]
## try to reproduce tuckman
Th_roll = xr.merge([Th,Ttrop]).rolling({"time":241},center=True)
Th_mean = Th_roll.mean().isel(time=slice(120,-120)).mean("member")
Th_max = Th_roll.max().isel(time=slice(120,-120)).mean("member")
Th_min = Th_roll.min().isel(time=slice(120,-120)).mean("member")
# Th_q = Th_q.rolling({"time":31}, center=True, min_periods=1).mean()

In [ ]:
v0="trop_sst_15"
v1 = "T_3_tot"

fig,ax = plt.subplots(figsize=(4,4))
ax.set_aspect("equal")

ax.scatter(
    sel(Th_mean[v0]), sel(Th_mean[v1]), s=10, c="k",
)


ax.scatter(
    sel(Th_mean[v0]), sel(Th_min[v1]), s=10,
)

ax.scatter(
    sel(Th_mean[v0]), sel(Th_max[v1]), s=10,
)

plt.show()

In [ ]:
v = "T_3_tot"

fig,axs = plt.subplots(1,2, figsize=(7,3))
# ax.plot(Th_q["T_3"].sel(quantile=.5))
for ax, y in zip(axs,[Th_q[v], np.abs((Th_q-Th_q.sel(quantile=.5))[v])]):
    
    ax.plot(y.time.dt.year, y.sel(quantile=.5), c="k")
    ax.plot(y.time.dt.year, y.sel(quantile=.05), label="La Niña")
    ax.plot(y.time.dt.year, y.sel(quantile=.95), label="El Niño")
    ax.axvline(2010, ls="--", c="k")
ax.legend()
plt.show()

Seasonal cycle

In [ ]:
x = Th.isel(time=slice(None,480))

get_q = lambda x : x.quantile(q=[.05,.5,.95], dim=["member","time"])

x_bm = x.groupby("time.month").map(get_q)

In [ ]:
plt.plot(x_bm.month, x_bm["T_3_tot"].sel(quantile=.05))
plt.plot(x_bm.month, x_bm["T_3_tot"].sel(quantile=.95))